# BrieFYI RAG 기능 테스트

현재 PostgreSQL의 `raw_articles`만 사용해 청킹, BGE-M3 임베딩, 인덱싱, 검색을 단계별로 확인한다.
- `RUN_INDEXING=True`일 때만 `article_chunks`, `chunk_embeddings`에 저장한다.


In [36]:
from pathlib import Path
from urllib.parse import urlparse
import math
import sys

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "rag":
    PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / "rag" / "retriever.py").exists():
    raise RuntimeError("BrieFYI 루트 또는 rag/ 디렉터리에서 노트북을 실행해 주세요.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from config import config
from db.db import get_conn
from rag.chunk import (
    CHUNK_OVERLAP_TOKENS,
    CHUNK_SIZE_TOKENS,
    build_article_text,
    split_text,
)
from rag.embed import embed_query
from rag.indexer import index_all_articles
from rag.retriever import retrieve

print(f"project root: {PROJECT_ROOT}")

project root: /home/thkim0/github/BrieFYI


## 1. 실행 범위 설정

아래 플래그를 필요한 단계에서만 `True`로 바꾼다. 벡터 및 hybrid 검색은 질의를 임베딩하므로 HF API를 호출한다.

In [48]:
RUN_HF_SMOKE = True # 1회 임베딩 확인 
RUN_INDEXING = False # 현재 db의 기사 인덱싱 
RUN_SEARCH = True # 검색
RUN_REINDEX_CHECK = True
RUN_INTERACTIVE = True

TOP_K = 5
TEST_QUERIES = [
    "AI 기술 동향",
    "인공지능 모델과 서비스",
]

pd.Series({
    "HF smoke": RUN_HF_SMOKE,
    "indexing": RUN_INDEXING,
    "search": RUN_SEARCH,
    "reindex check": RUN_REINDEX_CHECK,
    "interactive search": RUN_INTERACTIVE,
}, name="enabled")

HF smoke               True
indexing              False
search                 True
reindex check          True
interactive search     True
Name: enabled, dtype: bool

## 2. 설정 확인

비밀번호와 HF token은 출력하지 않고 설정 여부만 확인한다.

In [49]:
database = urlparse(config.DATABASE_URL)
safe_config = pd.Series({
    "database host": database.hostname,
    "database port": database.port,
    "database name": database.path.lstrip("/"),
    "HF token configured": bool(config.HF_TOKEN),
    "embedding model": config.HF_EMBEDDING_MODEL,
    "embedding dimension": config.HF_EMBEDDING_DIMENSION,
    "chunk size": CHUNK_SIZE_TOKENS,
    "chunk overlap": CHUNK_OVERLAP_TOKENS,
})
display(safe_config.to_frame("value"))

,value
database host,localhost
database port,5432
database name,briefyi
HF token configured,True
embedding model,BAAI/bge-m3
embedding dimension,1024
chunk size,500
chunk overlap,0


## 3. DB와 원본 기사 확인

pgvector 확장과 세 테이블을 확인한 뒤 현재 저장된 기사와 인덱스 건수를 조회한다.

In [50]:
required_tables = {"raw_articles", "article_chunks", "chunk_embeddings"}

with get_conn() as conn:
    identity = conn.execute(
        "SELECT current_database() AS database, current_user AS db_user"
    ).fetchone()
    extension = conn.execute(
        "SELECT extname, extversion FROM pg_extension WHERE extname = 'vector'"
    ).fetchone()
    table_rows = conn.execute(
        """SELECT table_name
           FROM information_schema.tables
           WHERE table_schema = 'public'
           ORDER BY table_name"""
    ).fetchall()

tables = {row["table_name"] for row in table_rows}
display(pd.Series(identity, name="value").to_frame())
display(pd.DataFrame([extension]))
display(pd.DataFrame(table_rows))
assert extension is not None, "pgvector 확장이 설치되어 있지 않습니다."
assert required_tables <= tables, f"필요한 테이블이 없습니다: {required_tables - tables}"

,value
database,briefyi
db_user,briefyi


,extname,extversion
0,vector,0.8.6


,table_name
0,article_chunks
1,chunk_embeddings
2,digests
3,raw_articles
4,send_log


In [51]:
with get_conn() as conn:
    counts = conn.execute(
        """SELECT
               (SELECT count(*) FROM raw_articles) AS raw_articles,
               (SELECT count(*) FROM article_chunks) AS article_chunks,
               (SELECT count(*) FROM chunk_embeddings) AS chunk_embeddings"""
    ).fetchone()
    article_rows = conn.execute(
        """SELECT id, title, source, published_at,
                  length(coalesce(description, '')) AS description_chars
           FROM raw_articles
           ORDER BY id"""
    ).fetchall()

display(pd.Series(counts, name="rows").to_frame())
display(pd.DataFrame(article_rows))

,rows
raw_articles,10
article_chunks,10
chunk_embeddings,10


,id,title,source,published_at,description_chars
0,19,Palantir: Winning AI Race With AIP And Forward...,Seeking Alpha,2026-08-14 18:51:38+00:00,140
1,20,"Claude Users Cancel Subscriptions, Citing Anth...",Business Insider,2026-08-14 18:48:37+00:00,144
2,21,S&P 500 Slips Amid High AI Stock Valuations an...,Devdiscourse,2026-08-14 18:47:52+00:00,341
3,22,Anthropic's Claude AI Model Fires Human Worker...,Sputnik News,2026-08-14 18:44:00+00:00,216
4,23,Punjab SSF Charts Roadmap for India’s First 3D...,Times Now,2026-08-14 18:37:34+00:00,197
5,24,Elon Musk deler bisarr AI-video: – Hvor mye?,Nettavisen,2026-08-14 18:36:31+00:00,71
6,25,Nielsen's DoubleVerify Deal Isn't About AI Ado...,Adweek,2026-08-14 18:33:25+00:00,91
7,26,AI Agent Was Asked To Book A Pilates Class. It...,Times Now,2026-08-14 18:32:30+00:00,204
8,27,"വിഡിയോ കോളിൽ വിവാഹം, നേരിൽ കണ്ടതോടെ വിവാഹമോചനം...",Malayala Manorama,2026-08-14 18:32:28+00:00,1081
9,28,This Acer AI Laptop is $500 Off and Packs Prem...,CNET,2026-08-14 18:32:17+00:00,105


## 4. 청킹 결과 미리보기

현재 HF API 방식은 로컬 tokenizer를 쓰지 않으므로 짧은 기사 하나가 청크 하나가 된다. overlap 코드는 구현되어 있지만 현재 값은 0이다.

In [52]:
with get_conn() as conn:
    preview_articles = conn.execute(
        """SELECT id, title, description
           FROM raw_articles
           ORDER BY id
           LIMIT 3"""
    ).fetchall()

chunk_preview = []
for article in preview_articles:
    article_text = build_article_text(article["title"], article["description"])
    for chunk in split_text(article_text):
        chunk_preview.append({
            "article_id": article["id"],
            "chunk_index": chunk["chunk_index"],
            "characters": len(chunk["chunk_text"]),
            "chunk_text": chunk["chunk_text"],
        })

display(pd.DataFrame(chunk_preview))

,article_id,chunk_index,characters,chunk_text
0,19,0,229,Palantir: Winning AI Race With AIP And Forward...
1,20,0,216,"Claude Users Cancel Subscriptions, Citing Anth..."
2,21,0,412,S&P 500 Slips Amid High AI Stock Valuations an...


## 5. BGE-M3 임베딩 단건 확인 (선택)

`RUN_HF_SMOKE=True`로 바꾸면 짧은 텍스트 한 건을 HF API에 보내 1024차원, 유한값, L2 정규화를 확인한다. 벡터 전체는 출력하지 않는다.

In [53]:
if RUN_HF_SMOKE:
    smoke_vector = embed_query("AI 기술 동향")
    smoke_norm = math.sqrt(sum(value * value for value in smoke_vector))
    smoke_result = pd.Series({
        "dimension": len(smoke_vector),
        "all finite": all(math.isfinite(value) for value in smoke_vector),
        "L2 norm": smoke_norm,
        "first 5 values": smoke_vector[:5],
    })
    display(smoke_result.to_frame("value"))
    assert len(smoke_vector) == config.HF_EMBEDDING_DIMENSION
    assert all(math.isfinite(value) for value in smoke_vector)
    assert abs(smoke_norm - 1.0) < 1e-5
else:
    print("Skipped: RUN_HF_SMOKE=False")

,value
dimension,1024
all finite,True
L2 norm,1.0
first 5 values,"[-0.04660012675313975, 0.0065237294452170375, ..."


## 6. 현재 기사 인덱싱 (선택, DB 쓰기)

`RUN_INDEXING=True`이면 현재 DB의 `raw_articles` 전체를 임베딩하고 저장한다. 외부 기사 데이터는 읽지 않는다.

In [54]:
if RUN_INDEXING:
    indexing_results = index_all_articles()
    display(pd.DataFrame(indexing_results))
else:
    print("Skipped: RUN_INDEXING=False")

Skipped: RUN_INDEXING=False


## 7. 저장 결과 무결성 확인

인덱싱 전에도 안전하게 실행할 수 있다. 모델별 개수와 차원, 고아 레코드 여부, 저장 벡터 norm을 확인한다.

In [55]:
with get_conn() as conn:
    index_counts = conn.execute(
        """SELECT
               (SELECT count(*) FROM article_chunks) AS article_chunks,
               (SELECT count(*) FROM chunk_embeddings) AS chunk_embeddings,
               (SELECT count(*)
                  FROM article_chunks ac
                  LEFT JOIN raw_articles ra ON ra.id = ac.article_id
                  WHERE ra.id IS NULL) AS orphan_chunks,
               (SELECT count(*)
                  FROM chunk_embeddings ce
                  LEFT JOIN article_chunks ac ON ac.id = ce.chunk_id
                  WHERE ac.id IS NULL) AS orphan_embeddings"""
    ).fetchone()
    model_rows = conn.execute(
        """SELECT embedding_model, embedding_dimension, count(*) AS rows,
                  avg(vector_norm(embedding)) AS average_l2_norm
           FROM chunk_embeddings
           GROUP BY embedding_model, embedding_dimension
           ORDER BY embedding_model, embedding_dimension"""
    ).fetchall()

display(pd.Series(index_counts, name="rows").to_frame())
display(pd.DataFrame(model_rows))
assert index_counts["orphan_chunks"] == 0
assert index_counts["orphan_embeddings"] == 0

,rows
article_chunks,10
chunk_embeddings,10
orphan_chunks,0
orphan_embeddings,0


,embedding_model,embedding_dimension,rows,average_l2_norm
0,BAAI/bge-m3,1024,10,1.0


## 8. vector / text / RRF hybrid 검색 비교 (선택)

- vector: BGE-M3 query embedding과 저장 벡터의 cosine 유사도
- text: PostgreSQL full-text 점수
- hybrid: 두 후보군의 dense rank를 weighted RRF로 결합

`RUN_SEARCH=True`이면 vector 및 hybrid 검색에서 HF API가 호출된다. 아래에 저장된 과거 normalized hybrid 출력이 있다면 이 셀을 다시 실행해 RRF 결과로 갱신한다.

In [56]:
SEARCH_COLUMNS = [
    "article_id", "chunk_id", "title", "score",
    "vector_score", "text_score",
    "vector_rank", "text_rank",
    "vector_rrf_score", "text_rrf_score", "text",
]

def search_frame(query, mode):
    rows = retrieve(
        query=query,
        top_k=TOP_K,
        search_mode=mode,
        metric="cosine",
        vector_weight=0.7,
        text_weight=0.3,
        fusion_method="rrf",
        candidate_k=50,
        rrf_k=60,
    )
    frame = pd.DataFrame(rows)
    if frame.empty:
        return frame
    return frame[[column for column in SEARCH_COLUMNS if column in frame.columns]]

if RUN_SEARCH:
    for test_query in TEST_QUERIES:
        print(f"query: {test_query}")
        for search_mode in ("vector", "text", "hybrid"):
            print(f"mode: {search_mode}")
            display(search_frame(test_query, search_mode))
else:
    print("Skipped: RUN_SEARCH=False")

query: AI 기술 동향
mode: vector


,article_id,chunk_id,title,score,vector_score,text
0,19,19,Palantir: Winning AI Race With AIP And Forward...,0.460638,0.460638,Palantir: Winning AI Race With AIP And Forward...
1,26,26,AI Agent Was Asked To Book A Pilates Class. It...,0.447281,0.447281,AI Agent Was Asked To Book A Pilates Class. It...
2,21,21,S&P 500 Slips Amid High AI Stock Valuations an...,0.438219,0.438219,S&P 500 Slips Amid High AI Stock Valuations an...
3,28,28,This Acer AI Laptop is $500 Off and Packs Prem...,0.426843,0.426843,This Acer AI Laptop is $500 Off and Packs Prem...
4,22,22,Anthropic's Claude AI Model Fires Human Worker...,0.425062,0.425062,Anthropic's Claude AI Model Fires Human Worker...


mode: text


""


mode: hybrid


...

,article_id,chunk_id,title,score,vector_score,text
0,22,22,Anthropic's Claude AI Model Fires Human Worker...,0.464465,0.464465,Anthropic's Claude AI Model Fires Human Worker...
1,26,26,AI Agent Was Asked To Book A Pilates Class. It...,0.456858,0.456858,AI Agent Was Asked To Book A Pilates Class. It...
2,25,25,Nielsen's DoubleVerify Deal Isn't About AI Ado...,0.434536,0.434536,Nielsen's DoubleVerify Deal Isn't About AI Ado...
3,28,28,This Acer AI Laptop is $500 Off and Packs Prem...,0.391295,0.391295,This Acer AI Laptop is $500 Off and Packs Prem...
4,23,23,Punjab SSF Charts Roadmap for India’s First 3D...,0.375810,0.375810,Punjab SSF Charts Roadmap for India’s First 3D...


mode: text


""


mode: hybrid


,article_id,chunk_id,title,score,vector_score,vector_score_normalized,text_score_normalized,text
0,22,22,Anthropic's Claude AI Model Fires Human Worker...,0.700000,0.464465,1.000000,0.0,Anthropic's Claude AI Model Fires Human Worker...
1,26,26,AI Agent Was Asked To Book A Pilates Class. It...,0.659922,0.456858,0.942746,0.0,AI Agent Was Asked To Book A Pilates Class. It...
2,25,25,Nielsen's DoubleVerify Deal Isn't About AI Ado...,0.542324,0.434536,0.774749,0.0,Nielsen's DoubleVerify Deal Isn't About AI Ado...
3,28,28,This Acer AI Laptop is $500 Off and Packs Prem...,0.314521,0.391295,0.449316,0.0,This Acer AI Laptop is $500 Off and Packs Prem...
4,23,23,Punjab SSF Charts Roadmap for India’s First 3D...,0.232943,0.375810,0.332775,0.0,Punjab SSF Charts Roadmap for India’s First 3D...


hybrid 결과에서는 원점수와 `vector_rank`, `text_rank`, `vector_rrf_score`, `text_rrf_score`, 최종 `score`를 함께 본다. 동일한 원점수에는 같은 dense rank를 부여하며, 한 검색기에 없는 후보의 해당 RRF 기여도는 0이다.

## 9. 재인덱싱 안정성 확인 (선택, DB 쓰기)

같은 모델로 다시 인덱싱했을 때 `(article_id, chunk_index)`와 `(chunk_id, embedding_model)` 기준으로 행 수가 불필요하게 늘어나지 않는지 확인한다. HF API를 다시 호출한다.

In [57]:
def current_index_counts():
    with get_conn() as conn:
        return conn.execute(
            """SELECT
                   (SELECT count(*) FROM article_chunks) AS article_chunks,
                   (SELECT count(*) FROM chunk_embeddings) AS chunk_embeddings"""
        ).fetchone()

if RUN_REINDEX_CHECK:
    before_reindex = current_index_counts()
    reindex_results = index_all_articles()
    after_reindex = current_index_counts()
    display(pd.DataFrame([before_reindex, after_reindex], index=["before", "after"]))
    display(pd.DataFrame(reindex_results))
    assert before_reindex == after_reindex
else:
    print("Skipped: RUN_REINDEX_CHECK=False")

,article_chunks,chunk_embeddings
before,10,10
after,10,10


,article_id,chunk_count,embedding_model,embedding_dimension
0,19,1,BAAI/bge-m3,1024
1,20,1,BAAI/bge-m3,1024
2,21,1,BAAI/bge-m3,1024
3,22,1,BAAI/bge-m3,1024
4,23,1,BAAI/bge-m3,1024
5,24,1,BAAI/bge-m3,1024
6,25,1,BAAI/bge-m3,1024
7,26,1,BAAI/bge-m3,1024
8,27,1,BAAI/bge-m3,1024
9,28,1,BAAI/bge-m3,1024


## 10. 직접 질의 입력 (선택)

`RUN_INTERACTIVE=True`일 때만 입력창을 연다. 전체 실행 시 멈추지 않도록 기본값은 `False`다.

In [58]:
if RUN_INTERACTIVE:
    user_query = input("검색어를 입력하세요: " ).strip()
    if not user_query:
        raise ValueError("검색어를 입력해 주세요.")
    display(search_frame(user_query, "hybrid"))
else:
    print("Skipped: RUN_INTERACTIVE=False")

,article_id,chunk_id,title,score,vector_score,text_score,vector_score_normalized,text_score_normalized,text
0,22,22,Anthropic's Claude AI Model Fires Human Worker...,1.000000,0.374930,0.3,1.000000,1.0,Anthropic's Claude AI Model Fires Human Worker...
1,27,27,"വിഡിയോ കോളിൽ വിവാഹം, നേരിൽ കണ്ടതോടെ വിവാഹമോചനം...",0.396409,0.292600,NaN,0.566299,0.0,"വിഡിയോ കോളിൽ വിവാഹം, നേരിൽ കണ്ടതോടെ വിവാഹമോചനം..."
2,20,20,"Claude Users Cancel Subscriptions, Citing Anth...",0.393283,0.291752,0.2,0.561833,0.0,"Claude Users Cancel Subscriptions, Citing Anth..."
3,26,26,AI Agent Was Asked To Book A Pilates Class. It...,0.333476,0.275533,NaN,0.476394,0.0,AI Agent Was Asked To Book A Pilates Class. It...
4,25,25,Nielsen's DoubleVerify Deal Isn't About AI Ado...,0.225196,0.246169,NaN,0.321709,0.0,Nielsen's DoubleVerify Deal Isn't About AI Ado...
